In [ ]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %md
# MAGIC # ETL Process for Customer Data
# MAGIC This notebook performs an ETL process to create a comprehensive customer profile by ingesting, transforming, and enriching data from various sources.

# COMMAND ----------
# MAGIC
# Configure logging
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, max, avg, expr, current_date, datediff

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Assume the Spark session is pre-initialized
# spark = SparkSession.builder.getOrCreate()

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 1: Data Ingestion
# MAGIC Load data from CSV files into DataFrames.

# COMMAND ----------
# MAGIC
try:
    logger.info("Loading data from CSV files into DataFrames.")
    policy_df = spark.read.csv("tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/policy.csv", header=True, inferSchema=True)
    claims_df = spark.read.csv("tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/claims.csv", header=True, inferSchema=True)
    demographics_df = spark.read.csv("tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/demographics.csv", header=True, inferSchema=True)
    scores_df = spark.read.csv("tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/scores.csv", header=True, inferSchema=True)
    aiml_insights_df = spark.read.csv("tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/aiml_insights.csv", header=True, inferSchema=True)
except Exception as e:
    logger.error(f"Error loading data: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 2: Data Selection
# MAGIC Select specific fields from each DataFrame.

# COMMAND ----------
# MAGIC
try:
    logger.info("Selecting specific fields from each DataFrame.")
    selected_demographics_df = demographics_df.select(
        "Customer_ID", "Customer_Name", "Email", "Phone_Number", "Address", "City", "State", "Postal_Code", 
        "Date_of_Birth", "Gender", "Marital_Status", "Occupation", "Income_Level", "Customer_Segment"
    )
    selected_claims_df = claims_df.select(
        "Claim_ID", "Policy_ID", "Claim_Date", "Claim_Type", "Claim_Status", "Claim_Amount", "Claim_Payout"
    )
    selected_policy_df = policy_df.select(
        "policy_id", "customer_id", "policy_type", "policy_status", "policy_start_date", "policy_end_date", 
        "policy_term", "policy_premium", "total_premium_paid", "renewal_status", "policy_addons"
    )
except Exception as e:
    logger.error(f"Error selecting data: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 3: Data Joining
# MAGIC Join datasets to create a unified view of customer data.

# COMMAND ----------
# MAGIC
try:
    logger.info("Joining datasets to create a unified view of customer data.")
    joined_df = selected_demographics_df.join(selected_policy_df, col("Customer_ID") == col("customer_id"), "inner") \
                                        .join(selected_claims_df, col("policy_id") == col("Policy_ID"), "inner")
except Exception as e:
    logger.error(f"Error joining data: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 4: Data Aggregation and Summarization
# MAGIC Aggregate claims data to calculate metrics.

# COMMAND ----------
# MAGIC
try:
    logger.info("Aggregating claims data to calculate metrics.")
    aggregated_df = joined_df.groupBy("Customer_ID").agg(
        count("Claim_ID").alias("Total_Claims"),
        count("policy_id").alias("Policy_Count"),
        max("Claim_Date").alias("Recent_Claim_Date"),
        avg("Claim_Amount").alias("Average_Claim_Amount")
    )
except Exception as e:
    logger.error(f"Error aggregating data: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 5: Custom Calculations
# MAGIC Implement custom calculations for additional insights.

# COMMAND ----------
# MAGIC
try:
    logger.info("Implementing custom calculations for additional insights.")
    # Refactor complex expressions into variables for clarity
    age_expr = expr("floor(datediff(current_date(), to_date(Date_of_Birth, 'yyyy-MM-dd')) / 365)")
    claim_to_premium_ratio_expr = expr("CASE WHEN total_premium_paid != 0 THEN Claim_Amount / total_premium_paid ELSE 0 END")
    claims_per_policy_expr = expr("CASE WHEN Policy_Count != 0 THEN Total_Claims / Policy_Count ELSE 0 END")

    enriched_df = aggregated_df.withColumn("Age", age_expr) \
                               .withColumn("Claim_To_Premium_Ratio", claim_to_premium_ratio_expr) \
                               .withColumn("Claims_Per_Policy", claims_per_policy_expr) \
                               .withColumn("Retention_Rate", expr("0.85")) \
                               .withColumn("Cross_Sell_Opportunities", expr("'Multi-Policy Discount, Home Coverage Add-on'")) \
                               .withColumn("Upsell_Potential", expr("'Premium Vehicle Coverage'"))
except Exception as e:
    logger.error(f"Error in custom calculations: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 6: Data Enrichment
# MAGIC Combine enriched data with AI/ML insights and scores for a comprehensive profile.

# COMMAND ----------
# MAGIC
try:
    logger.info("Combining enriched data with AI/ML insights and scores for a comprehensive profile.")
    final_df = enriched_df.join(aiml_insights_df, "Customer_ID", "inner") \
                          .join(scores_df, "Customer_ID", "inner")
except Exception as e:
    logger.error(f"Error enriching data: {e}")
    raise

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 7: Output Generation
# MAGIC Write the final comprehensive customer profile to a CSV file.

# COMMAND ----------
# MAGIC
try:
    logger.info("Writing the final comprehensive customer profile to a CSV file.")
    final_df.write.csv("tfs://dataeconomy-9k42/62457/uploads/62457/Customer_360.csv", header=True)
except Exception as e:
    logger.error(f"Error writing output: {e}")
    raise

logger.info("ETL process completed successfully.")
